In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout, Concatenate, Bidirectional
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print("TensorFlow version:", tf.__version__)
print("GPU Device:", tf.test.gpu_device_name())

Num GPUs Available:  1
TensorFlow version: 2.10.1
GPU Device: /device:GPU:0


In [3]:
# PARAMETERS
data_path = r"C:\Users\ronle\Desktop\BUAS\Y2C\datasets\all_groups\group_combined_features.csv"
test_dataset_path = r"C:\Users\ronle\Desktop\BUAS\Y2C\datasets\all_groups\testing_dataset_g21_mapped_processed_featured.csv"
glove_path = r"C:\Users\ronle\Desktop\BUAS\Y2C\glove.6B.300d.txt"
max_words = 10000
max_len = 100
embedding_dim = 300

In [4]:
df = pd.read_csv(data_path)


In [5]:
# Examine the unique values in the emotion column to understand what we're working with
print("Unique emotions:", df['emotion'].unique())

# Create one-hot encoding for the emotions
emotions_dummies = pd.get_dummies(df['emotion'])
print("One-hot encoded emotion columns:", emotions_dummies.columns.tolist())

# Add these columns to the dataframe
df = pd.concat([df, emotions_dummies], axis=1)

# Update emotion_cols to match the created dummy columns
emotion_cols = emotions_dummies.columns.tolist()
print("Using emotion columns:", emotion_cols)

# Now create the label and one-hot encoded target
df['label'] = df['emotion_code']  # You can use emotion_code directly if it matches your needs
# Or if you need to create labels from the one-hot encoded columns:
# df['label'] = df[emotion_cols].values.argmax(axis=1)

num_classes = len(emotion_cols)
y = to_categorical(df['label'], num_classes=num_classes)

Unique emotions: ['surprise' 'neutral' 'anger' 'sadness' 'happiness' 'fear' 'disgust']
One-hot encoded emotion columns: ['anger', 'disgust', 'fear', 'happiness', 'neutral', 'sadness', 'surprise']
Using emotion columns: ['anger', 'disgust', 'fear', 'happiness', 'neutral', 'sadness', 'surprise']


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24295 entries, 0 to 24294
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   text                   24295 non-null  object 
 1   emotion                24295 non-null  object 
 2   emotion_code           24295 non-null  int64  
 3   sub_emotion            24295 non-null  object 
 4   TextBlob_Polarity      24295 non-null  float64
 5   TextBlob_Subjectivity  24295 non-null  float64
 6   VADER_Compound         24295 non-null  float64
 7   VADER_Positive         24295 non-null  float64
 8   VADER_Neutral          24295 non-null  float64
 9   VADER_Negative         24295 non-null  float64
 10  anger                  24295 non-null  bool   
 11  disgust                24295 non-null  bool   
 12  fear                   24295 non-null  bool   
 13  happiness              24295 non-null  bool   
 14  neutral                24295 non-null  bool   
 15  sa

In [7]:
# Check alignment between emotion_code and our one-hot encoding
for emotion, code in zip(['anger', 'disgust', 'fear', 'happiness', 'neutral', 'sadness', 'surprise'], range(7)):
    print(f"Emotion: {emotion}, Code: {code}")
    print(f"Sample count in dataset: {df[df['emotion'] == emotion].shape[0]}")
    print(f"Emotion code values: {df[df['emotion'] == emotion]['emotion_code'].unique()}")
    print("-" * 30)

# Decide whether to use the existing emotion_code or our one-hot encoding
# If emotion_code is already properly aligned (0-6), we can use it directly:
df['label'] = df['emotion_code']
# Otherwise, we need to map to our new ordering:
# emotion_mapping = {'anger': 0, 'disgust': 1, 'fear': 2, 'happiness': 3, 'neutral': 4, 'sadness': 5, 'surprise': 6}
# df['label'] = df['emotion'].map(emotion_mapping)

num_classes = len(emotion_cols)
y = to_categorical(df['label'], num_classes=num_classes)

Emotion: anger, Code: 0
Sample count in dataset: 1473
Emotion code values: [2]
------------------------------
Emotion: disgust, Code: 1
Sample count in dataset: 150
Emotion code values: [5]
------------------------------
Emotion: fear, Code: 2
Sample count in dataset: 775
Emotion code values: [3]
------------------------------
Emotion: happiness, Code: 3
Sample count in dataset: 3260
Emotion code values: [0]
------------------------------
Emotion: neutral, Code: 4
Sample count in dataset: 11777
Emotion code values: [6]
------------------------------
Emotion: sadness, Code: 5
Sample count in dataset: 990
Emotion code values: [1]
------------------------------
Emotion: surprise, Code: 6
Sample count in dataset: 5870
Emotion code values: [4]
------------------------------


In [8]:
# Keep your original text preprocessing steps
# TOKENIZE TEXT
df['text'] = df['text'].astype(str)
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df['text'])
sequences = tokenizer.texts_to_sequences(df['text'])
X_text = pad_sequences(sequences, maxlen=max_len)

# SPLIT DATA (80% train, 20% val) - modify this to only use text
X_text_train, X_text_val, y_train, y_val = train_test_split(
    X_text, y, test_size=0.20, random_state=42, stratify=y)


In [9]:
# LOAD PRETRAINED GLOVE EMBEDDINGS
embeddings_index = {}
with open(glove_path, encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coeffs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coeffs
print(f'Found {len(embeddings_index)} word vectors in GloVe.')

Found 400000 word vectors in GloVe.


In [10]:
# CREATE EMBEDDING MATRIX
word_index = tokenizer.word_index
num_words = min(max_words, len(word_index) + 1)
embedding_matrix = np.zeros((num_words, embedding_dim))
for word, i in word_index.items():
    if i >= max_words:
        continue
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

In [11]:

# DEFINE NEW MODEL ARCHITECTURE - TEXT ONLY
# Text branch
text_input = Input(shape=(max_len,), name='text_input')
embedding_layer = Embedding(input_dim=num_words,
                           output_dim=embedding_dim,
                           weights=[embedding_matrix],
                           input_length=max_len,
                           trainable=False)(text_input)

# LSTM layer 
lstm_out = Bidirectional(LSTM(128, 
                             dropout=0.2,
                             return_sequences=False,
                             unroll=False,
                             use_bias=True))(embedding_layer)

# Add dense layers
text_features = Dense(128, activation='relu')(lstm_out)
text_features = Dropout(0.3)(text_features)
dense = Dense(64, activation='relu')(text_features)
dense = Dropout(0.5)(dense)
output = Dense(num_classes, activation='softmax')(dense)

# Build and compile the text-only model
text_only_model = Model(inputs=text_input, outputs=output)
text_only_model.compile(optimizer=Adam(learning_rate=0.001),
                       loss='categorical_crossentropy',
                       metrics=['accuracy'])

text_only_model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 text_input (InputLayer)     [(None, 100)]             0         
                                                                 
 embedding (Embedding)       (None, 100, 300)          2822400   
                                                                 
 bidirectional (Bidirectiona  (None, 256)              439296    
 l)                                                              
                                                                 
 dense (Dense)               (None, 128)               32896     
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense_1 (Dense)             (None, 64)                8256      
                                                             

In [12]:
# DEFINE CALLBACKS
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ModelCheckpoint(filepath='best_lstm_model.h5', monitor='val_loss', save_best_only=True)
]


In [13]:
# TRAIN THE TEXT-ONLY MODEL
history_text_only = text_only_model.fit(
    X_text_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_text_val, y_val),
    callbacks=callbacks
)

Epoch 1/50
608/608 [==============================] - 48s 62ms/step - loss: 1.3336 - accuracy: 0.5119 - val_loss: 1.1743 - val_accuracy: 0.5678
Epoch 2/50
608/608 [==============================] - 36s 60ms/step - loss: 1.1858 - accuracy: 0.5669 - val_loss: 1.1152 - val_accuracy: 0.5907
Epoch 3/50
608/608 [==============================] - 36s 59ms/step - loss: 1.1154 - accuracy: 0.5970 - val_loss: 1.0972 - val_accuracy: 0.5968
Epoch 4/50
608/608 [==============================] - 35s 58ms/step - loss: 1.0604 - accuracy: 0.6201 - val_loss: 1.0888 - val_accuracy: 0.6073
Epoch 5/50
608/608 [==============================] - 30s 49ms/step - loss: 1.0068 - accuracy: 0.6366 - val_loss: 1.0709 - val_accuracy: 0.6166
Epoch 6/50
608/608 [==============================] - 29s 47ms/step - loss: 0.9377 - accuracy: 0.6652 - val_loss: 1.0740 - val_accuracy: 0.6244
Epoch 7/50
608/608 [==============================] - 29s 48ms/step - loss: 0.8885 - accuracy: 0.6797 - val_loss: 1.0727 - val_accuracy:

In [14]:
test_df = pd.read_csv(test_dataset_path)


In [15]:
# Load and process test dataset
test_df = pd.read_csv(test_dataset_path)
print("Test dataset columns:", test_df.columns.tolist())

Test dataset columns: ['POS_Tags', 'TF_IDF', 'TextBlob_Polarity', 'TextBlob_Scores', 'TextBlob_Sentiment', 'TextBlob_Subjectivity', 'VADER_Compound', 'VADER_Negative', 'VADER_Neutral', 'VADER_Positive', 'VADER_Scores', 'anger', 'disgust', 'emotion', 'emotion_code', 'fear', 'happiness', 'neutral', 'sadness', 'surprise', 'text']


In [16]:
# If the test dataset has the same structure:
test_df['label'] = test_df['emotion_code']  # Use the same approach as training data
y_test = to_categorical(test_df['label'], num_classes=num_classes)

In [17]:
# Tokenize and pad the test text data
test_df['text'] = test_df['text'].astype(str)
test_sequences = tokenizer.texts_to_sequences(test_df['text'])
X_text_test = pad_sequences(test_sequences, maxlen=max_len)

In [19]:
# EVALUATE ON TEST SET
# Prepare test data (only text part)
test_df['text'] = test_df['text'].astype(str)
test_sequences = tokenizer.texts_to_sequences(test_df['text'])
X_text_test = pad_sequences(test_sequences, maxlen=max_len)

print("\n--- Evaluating Text-Only Model on Test Set ---")
loss_text_only, acc_text_only = text_only_model.evaluate(X_text_test, y_test)
print(f'Text-Only Model Test Loss: {loss_text_only:.4f} / Test Accuracy: {acc_text_only:.4f}')

# Get predictions for metrics
y_pred_text_only = text_only_model.predict(X_text_test)
y_pred_text_only_classes = np.argmax(y_pred_text_only, axis=1)
y_true = np.argmax(y_test, axis=1)

# Calculate metrics
accuracy = accuracy_score(y_true, y_pred_text_only_classes)
precision = precision_score(y_true, y_pred_text_only_classes, average='weighted')
recall = recall_score(y_true, y_pred_text_only_classes, average='weighted')
f1 = f1_score(y_true, y_pred_text_only_classes, average='weighted')

# Print overall metrics
print("\nText-Only Model Metrics:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

# If you want to compare with original model, add a section like:
print("\n--- Comparison between Models ---")
print(f"Original LSTM (with sentiment): Accuracy={accuracy:.4f}")
print(f"Text-only LSTM: Accuracy={acc_text_only:.4f}")
print(f"Difference: {(accuracy-acc_text_only)*100:.2f}% points")


--- Evaluating Text-Only Model on Test Set ---
27/27 [==============================] - 1s 21ms/step - loss: 1.9168 - accuracy: 0.4058
Text-Only Model Test Loss: 1.9168 / Test Accuracy: 0.4058
27/27 [==============================] - 0s 13ms/step

Text-Only Model Metrics:
Accuracy: 0.4058
Precision: 0.4999
Recall: 0.4058
F1 Score: 0.3763

--- Comparison between Models ---
Original LSTM (with sentiment): Accuracy=0.4058
Text-only LSTM: Accuracy=0.4058
Difference: -0.00% points


c:\Users\ronle\anaconda3\envs\y2c\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Get predictions and true labels
y_pred_lstm = lstm_model.predict({'text_input': X_text_test, 'sentiment_input': test_sentiment_features})
y_pred_lstm_classes = np.argmax(y_pred_lstm, axis=1)
y_true = np.argmax(y_test, axis=1)

In [ ]:
# Calculate metrics
accuracy = accuracy_score(y_true, y_pred_lstm_classes)
precision = precision_score(y_true, y_pred_lstm_classes, average='weighted')
recall = recall_score(y_true, y_pred_lstm_classes, average='weighted')
f1 = f1_score(y_true, y_pred_lstm_classes, average='weighted')

In [ ]:
# Print overall metrics
print("\nOverall Metrics:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

In [ ]:
# Learning Curves
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history_lstm.history['loss'], label='Train Loss')
plt.plot(history_lstm.history['val_loss'], label='Validation Loss')
plt.title('LSTM Loss Curves')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_lstm.history['accuracy'], label='Train Accuracy')
plt.plot(history_lstm.history['val_accuracy'], label='Validation Accuracy')
plt.title('LSTM Accuracy Curves')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.savefig('LSTM_9_learning_curves.png')
plt.show()

In [ ]:

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_lstm_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
           xticklabels=emotion_cols, yticklabels=emotion_cols)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('LSTM Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Classification Report
print("\nLSTM Classification Report:")
report = classification_report(y_true, y_pred_lstm_classes, target_names=emotion_cols, output_dict=True)
print(classification_report(y_true, y_pred_lstm_classes, target_names=emotion_cols))

In [ ]:
# 4. Per-class metrics visualization
plt.figure(figsize=(12, 6))
metrics_df = pd.DataFrame(report).transpose()
metrics_df = metrics_df.iloc[:-3]  # Remove avg rows
metrics_df = metrics_df[['precision', 'recall', 'f1-score']]
metrics_df.plot(kind='bar')
plt.title('Per-Class Metrics')
plt.ylabel('Score')
plt.xlabel('Emotion')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 6. Additional: Prediction distribution
plt.figure(figsize=(10, 6))
pd.Series(y_pred_lstm_classes).map(dict(enumerate(emotion_cols))).value_counts().plot(kind='bar')
plt.title('Distribution of Predicted Emotions')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# 7. Compare with true distribution
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
pd.Series(y_true).map(dict(enumerate(emotion_cols))).value_counts().plot(kind='bar')
plt.title('True Emotion Distribution')
plt.xlabel('Emotion')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
pd.Series(y_pred_lstm_classes).map(dict(enumerate(emotion_cols))).value_counts().plot(kind='bar')
plt.title('Predicted Emotion Distribution')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [32]:
# PARAMETERS
data_path = r"C:\Users\ronle\Desktop\BUAS\Y2C\datasets\all_groups\group_combined_features.csv"
test_dataset_path = r"C:\Users\ronle\Desktop\BUAS\Y2C\datasets\all_groups\testing_dataset_g21_mapped_processed_featured.csv"
glove_path = r"C:\Users\ronle\Desktop\BUAS\Y2C\glove.6B.300d.txt"
max_words = 10000
max_len = 100
embedding_dim = 300

In [ ]:
df = pd.read_csv(data_path)
df


In [ ]:
df = pd.read_csv(test_dataset_path)
df['text'].head(100)

# To display all rows in this specific Series
print(df['text'].to_string())

# OR change pandas display settings globally
import pandas as pd
pd.set_option('display.max_rows', 100)  # Show up to 100 rows
# pd.set_option('display.max_rows', None)  # Show all rows regardless of number


In [ ]:
# To display all rows in this specific Series
print(df['text'].to_string())

# OR change pandas display settings globally
import pandas as pd
pd.set_option('display.max_rows', 100)  # Show up to 100 rows
# pd.set_option('display.max_rows', None)  # Show all rows regardless of number